In [49]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot(title, label, x, result, expected):
    plt.plot(x, expected, label=f'Ground truth')
    plt.plot(x, result, label=f'Sequre {label}')
    plt.title(title)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)
    plt.show()

def mae(result, expected):
    return np.mean(np.abs(np.array(result) - np.array(expected)))

In [50]:
""" Core methods """

# Until Codon Jupyter is fixed: Read the data from files
show_plots = False

dump_folder = "dump"
dump_files = [
    "decor_trig",
    "decor",
    "fourier",
    "cheby",
    "taylor"
    ]
nbit_fs = [32]
intervals_count = 4
cps = [0, 1]

df_data = {
    'Method': [],
    'Interval': [],
    'MAE': [],
    'Runtime': [],
    'Partitions count': [],
    'Truncations count': []
    }

for cp in cps:
    df_data[f'Bytes sent CP{cp}'] = []
    df_data[f'Requests sent CP{cp}'] = []

for dump_file in dump_files:
    for nbit_f in nbit_fs:
        for i in range(intervals_count):
            for cp in cps:
                try:
                    with open(f"{dump_folder}/{dump_file}_{i}_{nbit_f}_CP{cp}.p", "rb") as f:
                        data = pickle.load(f)
                        x = data['x']
                        interval = f"({round(min(x), 2)}, {round(max(x), 2)})"
                        for k, v in data.items():
                            if not k.endswith('_result'):
                                continue
                            
                            k = k.replace('_result', '')
                            expected = data[f"{k}_expected"]

                            runtime = round(data[f"{k}_time"][0], 5)
                            bytes_sent = int(data[f"{k}_bytes_sent"][0])
                            send_requests = int(data[f"{k}_send_requests"][0])
                            partitions_count = int(data[f"{k}_partitions_count"][0])
                            truncations_count = int(data[f"{k}_truncations_count"][0])
                            
                            if cp == 1:
                                df_data['Method'].append(f"{k}_{nbit_f}")
                                df_data['Interval'].append(interval)
                                df_data['MAE'].append(mae(v, expected))
                                
                                df_data['Runtime'].append(runtime)
                                df_data['Partitions count'].append(partitions_count)
                                df_data['Truncations count'].append(truncations_count)

                            df_data[f'Bytes sent CP{cp}'].append(bytes_sent)
                            df_data[f'Requests sent CP{cp}'].append(send_requests)
                            
                            if show_plots and cp == 1:
                                plot(f"{k} {nbit_f} frac bits on {interval}", k, x, v, expected)
                except FileNotFoundError:
                    print(f"Could not find {dump_folder}/{dump_file}_{i}_{nbit_f}.p")

df = pd.DataFrame(df_data)

In [51]:
def by_interval(df):
    for interval, group in df.groupby('Interval'):
        display(group)

In [52]:
by_interval(df[df['Method'].str.contains('sin')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
3,decor_sin_32,"(-1.0, 2.5)",5.779310e-10,0.00247,2,1,9664,8,4832,4
70,chebyshev_sin_naive_32,"(-1.0, 2.5)",5.135593e-09,0.02697,10,12,82160,68,53160,44
72,chebyshev_sin_clenshaw_32,"(-1.0, 2.5)",5.175561e-09,0.02895,10,12,82144,68,53152,44
66,chebyshev_sin_decor_32,"(-1.0, 2.5)",9.524740e-09,0.01156,24,4,66968,50,36624,58
128,taylor_sin_naive_32,"(-1.0, 2.5)",9.445958e-04,0.01484,6,7,48336,40,31416,26
133,taylor_sin_decor_32,"(-1.0, 2.5)",9.445983e-04,0.01083,24,3,52504,46,34208,56
30,fourier_sin_32,"(-1.0, 2.5)",5.198747e-02,0.00256,2,1,96384,8,26512,4


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
25,fourier_sin_32,"(-3.14, 3.14)",5.277295e-10,0.00229,2,1,9696,8,4840,4
1,decor_sin_32,"(-3.14, 3.14)",5.436982e-10,0.00248,2,1,9664,8,4832,4
55,chebyshev_sin_naive_32,"(-3.14, 3.14)",3.822418e-06,0.02512,10,12,82160,68,53160,44
57,chebyshev_sin_clenshaw_32,"(-3.14, 3.14)",3.822518e-06,0.02794,10,12,82144,68,53152,44
51,chebyshev_sin_decor_32,"(-3.14, 3.14)",3.823280e-06,0.01332,24,4,66968,50,36624,58
118,taylor_sin_naive_32,"(-3.14, 3.14)",7.196224e-02,0.01516,6,7,48336,40,31416,26
123,taylor_sin_decor_32,"(-3.14, 3.14)",7.196225e-02,0.01187,24,3,52504,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
7,decor_sin_32,"(-5.0, 5.0)",5.577745e-10,0.00237,2,1,9664,8,4832,4
105,chebyshev_sin_decor_32,"(-5.0, 5.0)",4.657796e-04,0.01363,24,4,66968,50,36624,58
109,chebyshev_sin_naive_32,"(-5.0, 5.0)",4.659120e-04,0.02695,10,12,82160,68,53160,44
111,chebyshev_sin_clenshaw_32,"(-5.0, 5.0)",4.659124e-04,0.02815,10,12,82144,68,53152,44
43,fourier_sin_32,"(-5.0, 5.0)",6.993225e-02,0.00253,2,1,96384,8,26512,4
159,taylor_sin_decor_32,"(-5.0, 5.0)",1.587863e+00,0.01079,24,3,52504,46,34208,56
154,taylor_sin_naive_32,"(-5.0, 5.0)",1.587863e+00,0.01496,6,7,48336,40,31416,26


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
92,chebyshev_sin_naive_32,"(0.79, 2.36)",1.593362e-10,0.02845,10,12,82160,68,53160,44
5,decor_sin_32,"(0.79, 2.36)",3.407539e-10,0.00223,2,1,9664,8,4832,4
95,chebyshev_sin_clenshaw_32,"(0.79, 2.36)",4.867687e-10,0.02631,10,12,82144,68,53152,44
83,chebyshev_sin_decor_32,"(0.79, 2.36)",4.786443e-09,0.01268,24,4,66968,50,36624,58
145,taylor_sin_naive_32,"(0.79, 2.36)",4.294456e-07,0.01631,6,7,48336,40,31416,26
148,taylor_sin_decor_32,"(0.79, 2.36)",4.294827e-07,0.01048,24,3,52504,46,34208,56
35,fourier_sin_32,"(0.79, 2.36)",9.267859e-04,0.00258,2,1,96384,8,26512,4


In [53]:
by_interval(df[df['Method'].str.contains('cos')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
2,decor_cos_32,"(-1.0, 2.5)",4.416170e-10,0.00222,2,1,9664,8,4832,4
64,chebyshev_cos_clenshaw_32,"(-1.0, 2.5)",4.734071e-09,0.02708,10,12,82144,68,53152,44
71,chebyshev_cos_naive_32,"(-1.0, 2.5)",4.786893e-09,0.02577,10,12,82160,68,53160,44
76,chebyshev_cos_decor_32,"(-1.0, 2.5)",1.755757e-08,0.01406,24,4,66968,50,36624,58
134,taylor_cos_naive_32,"(-1.0, 2.5)",8.799819e-04,0.01481,6,7,48336,40,31416,26
129,taylor_cos_decor_32,"(-1.0, 2.5)",8.799846e-04,0.01090,24,3,52504,46,34208,56
29,fourier_cos_32,"(-1.0, 2.5)",4.844328e-02,0.00206,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
24,fourier_cos_32,"(-3.14, 3.14)",5.317375e-10,0.00164,1,1,9696,8,2424,2
0,decor_cos_32,"(-3.14, 3.14)",5.420289e-10,0.00245,2,1,9664,8,4832,4
56,chebyshev_cos_naive_32,"(-3.14, 3.14)",5.237946e-07,0.02582,10,12,82160,68,53160,44
49,chebyshev_cos_clenshaw_32,"(-3.14, 3.14)",5.238901e-07,0.02738,10,12,82144,68,53152,44
61,chebyshev_cos_decor_32,"(-3.14, 3.14)",5.267609e-07,0.01212,24,4,66968,50,36624,58
119,taylor_cos_decor_32,"(-3.14, 3.14)",2.587264e-02,0.01089,24,3,52504,46,34208,56
124,taylor_cos_naive_32,"(-3.14, 3.14)",2.587297e-02,0.01476,6,7,48336,40,31416,26


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
6,decor_cos_32,"(-5.0, 5.0)",4.945201e-10,0.00236,2,1,9664,8,4832,4
115,chebyshev_cos_decor_32,"(-5.0, 5.0)",1.044081e-04,0.01357,24,4,66968,50,36624,58
110,chebyshev_cos_naive_32,"(-5.0, 5.0)",1.048419e-04,0.02586,10,12,82160,68,53160,44
103,chebyshev_cos_clenshaw_32,"(-5.0, 5.0)",1.048423e-04,0.02972,10,12,82144,68,53152,44
42,fourier_cos_32,"(-5.0, 5.0)",8.133451e-03,0.00231,1,1,96384,8,24096,2
155,taylor_cos_decor_32,"(-5.0, 5.0)",9.343857e-01,0.01030,24,3,52504,46,34208,56
160,taylor_cos_naive_32,"(-5.0, 5.0)",9.343927e-01,0.01502,6,7,48336,40,31416,26


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
93,chebyshev_cos_naive_32,"(0.79, 2.36)",2.532582e-10,0.02777,10,12,82160,68,53160,44
79,chebyshev_cos_clenshaw_32,"(0.79, 2.36)",4.678312e-10,0.02820,10,12,82144,68,53152,44
4,decor_cos_32,"(0.79, 2.36)",6.981197e-10,0.00220,2,1,9664,8,4832,4
100,chebyshev_cos_decor_32,"(0.79, 2.36)",3.040456e-09,0.01253,24,4,66968,50,36624,58
146,taylor_cos_decor_32,"(0.79, 2.36)",4.866542e-06,0.01036,24,3,52504,46,34208,56
149,taylor_cos_naive_32,"(0.79, 2.36)",4.866624e-06,0.01480,6,7,48336,40,31416,26
34,fourier_cos_32,"(0.79, 2.36)",5.095119e-02,0.00141,1,1,96384,8,24096,2


In [54]:
by_interval(df[df['Method'].str.contains('exp')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
13,decor_exp_32,"(-1.0, 2.5)",2.930874e-09,0.01017,24,3,38048,46,34208,56
75,chebyshev_exp_naive_32,"(-1.0, 2.5)",1.678105e-08,0.02608,10,12,82160,68,53160,44
62,chebyshev_exp_clenshaw_32,"(-1.0, 2.5)",1.678338e-08,0.02830,10,12,82144,68,53152,44
65,chebyshev_exp_decor_32,"(-1.0, 2.5)",2.420067e-08,0.01182,24,4,66968,50,36624,58
130,taylor_exp_naive_32,"(-1.0, 2.5)",2.929406e-03,0.01670,6,7,48336,40,31416,26
135,taylor_exp_decor_32,"(-1.0, 2.5)",2.929417e-03,0.01010,24,3,52504,46,34208,56
33,fourier_exp_32,"(-1.0, 2.5)",4.253193e-01,0.00227,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
9,decor_exp_32,"(-3.14, 3.14)",2.585457e-08,0.00976,24,3,38048,46,34208,56
60,chebyshev_exp_naive_32,"(-3.14, 3.14)",5.658436e-06,0.02651,10,12,82160,68,53160,44
47,chebyshev_exp_clenshaw_32,"(-3.14, 3.14)",5.658478e-06,0.02854,10,12,82144,68,53152,44
50,chebyshev_exp_decor_32,"(-3.14, 3.14)",5.658911e-06,0.01230,24,4,66968,50,36624,58
120,taylor_exp_naive_32,"(-3.14, 3.14)",8.999560e-02,0.01566,6,7,48336,40,31416,26
125,taylor_exp_decor_32,"(-3.14, 3.14)",8.999565e-02,0.01115,24,3,52504,46,34208,56
28,fourier_exp_32,"(-3.14, 3.14)",8.301076e-01,0.00220,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
21,decor_exp_32,"(-5.0, 5.0)",3.579387e-07,0.01055,24,3,38048,46,34208,56
101,chebyshev_exp_clenshaw_32,"(-5.0, 5.0)",1.259335e-03,0.02767,10,12,82144,68,53152,44
114,chebyshev_exp_naive_32,"(-5.0, 5.0)",1.259340e-03,0.02593,10,12,82160,68,53160,44
104,chebyshev_exp_decor_32,"(-5.0, 5.0)",1.259403e-03,0.01636,24,4,66968,50,36624,58
161,taylor_exp_decor_32,"(-5.0, 5.0)",2.796561e+00,0.01019,24,3,52504,46,34208,56
156,taylor_exp_naive_32,"(-5.0, 5.0)",2.796562e+00,0.01723,6,7,48336,40,31416,26
46,fourier_exp_32,"(-5.0, 5.0)",5.321508e+00,0.00223,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
17,decor_exp_32,"(0.79, 2.36)",1.651516e-09,0.00886,24,3,38048,46,34208,56
77,chebyshev_exp_clenshaw_32,"(0.79, 2.36)",1.707680e-09,0.02914,10,12,82144,68,53152,44
99,chebyshev_exp_naive_32,"(0.79, 2.36)",1.894699e-09,0.02740,10,12,82160,68,53160,44
82,chebyshev_exp_decor_32,"(0.79, 2.36)",9.939596e-09,0.01273,24,4,66968,50,36624,58
139,taylor_exp_naive_32,"(0.79, 2.36)",2.374040e-05,0.01512,6,7,48336,40,31416,26
144,taylor_exp_decor_32,"(0.79, 2.36)",2.374124e-05,0.01175,24,3,52504,46,34208,56
38,fourier_exp_32,"(0.79, 2.36)",3.009865e-01,0.00219,1,1,96384,8,24096,2


In [55]:
by_interval(df[df['Method'].str.contains('sigmoid')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
12,decor_sigmoid_32,"(-1.0, 2.5)",8.760171e-10,0.08403,186,25,282048,346,277216,432
67,chebyshev_sigmoid_naive_32,"(-1.0, 2.5)",1.700802e-07,0.02540,10,12,82160,68,53160,44
73,chebyshev_sigmoid_clenshaw_32,"(-1.0, 2.5)",1.701430e-07,0.02813,10,12,82144,68,53152,44
63,chebyshev_sigmoid_decor_32,"(-1.0, 2.5)",1.714565e-07,0.01366,24,4,66968,50,36624,58
126,taylor_sigmoid_decor_32,"(-1.0, 2.5)",8.343982e-03,0.01128,24,3,45280,46,34208,56
127,taylor_sigmoid_naive_32,"(-1.0, 2.5)",8.343982e-03,0.00814,3,4,26592,22,16920,14
31,fourier_sigmoid_32,"(-1.0, 2.5)",2.360811e-02,0.00257,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
8,decor_sigmoid_32,"(-3.14, 3.14)",4.358965e-09,0.08309,186,25,282048,346,277216,432
48,chebyshev_sigmoid_decor_32,"(-3.14, 3.14)",3.948888e-05,0.01759,24,4,66968,50,36624,58
52,chebyshev_sigmoid_naive_32,"(-3.14, 3.14)",3.949278e-05,0.02506,10,12,82160,68,53160,44
58,chebyshev_sigmoid_clenshaw_32,"(-3.14, 3.14)",3.949279e-05,0.02775,10,12,82144,68,53152,44
116,taylor_sigmoid_decor_32,"(-3.14, 3.14)",2.176966e-02,0.00992,24,3,45280,46,34208,56
117,taylor_sigmoid_naive_32,"(-3.14, 3.14)",2.176966e-02,0.00811,3,4,26592,22,16920,14
26,fourier_sigmoid_32,"(-3.14, 3.14)",3.304574e-02,0.00393,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
20,decor_sigmoid_32,"(-5.0, 5.0)",2.304810e-07,0.08217,186,25,282048,346,277216,432
106,chebyshev_sigmoid_naive_32,"(-5.0, 5.0)",7.580360e-04,0.02609,10,12,82160,68,53160,44
112,chebyshev_sigmoid_clenshaw_32,"(-5.0, 5.0)",7.580361e-04,0.02847,10,12,82144,68,53152,44
102,chebyshev_sigmoid_decor_32,"(-5.0, 5.0)",7.580565e-04,0.01486,24,4,66968,50,36624,58
44,fourier_sigmoid_32,"(-5.0, 5.0)",3.554065e-02,0.00253,1,1,96384,8,24096,2
153,taylor_sigmoid_naive_32,"(-5.0, 5.0)",2.192391e-01,0.00814,3,4,26592,22,16920,14
152,taylor_sigmoid_decor_32,"(-5.0, 5.0)",2.192391e-01,0.00977,24,3,45280,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
85,chebyshev_sigmoid_naive_32,"(0.79, 2.36)",2.318960e-10,0.02726,10,12,82160,68,53160,44
16,decor_sigmoid_32,"(0.79, 2.36)",2.434097e-10,0.08151,186,25,282048,346,277216,432
96,chebyshev_sigmoid_clenshaw_32,"(0.79, 2.36)",3.973031e-10,0.02852,10,12,82144,68,53152,44
78,chebyshev_sigmoid_decor_32,"(0.79, 2.36)",7.254925e-09,0.01531,24,4,66968,50,36624,58
136,taylor_sigmoid_decor_32,"(0.79, 2.36)",2.209328e-04,0.01122,24,3,45280,46,34208,56
137,taylor_sigmoid_naive_32,"(0.79, 2.36)",2.209329e-04,0.00802,3,4,26592,22,16920,14
36,fourier_sigmoid_32,"(0.79, 2.36)",8.161617e-03,0.00213,1,1,96384,8,24096,2


In [56]:
by_interval(df[df['Method'].str.contains('sqrt')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
98,chebyshev_sqrt_clenshaw_32,"(0.79, 2.36)",5.922684e-09,0.02823,10,12,82144,68,53152,44
94,chebyshev_sqrt_naive_32,"(0.79, 2.36)",5.995495e-09,0.02726,10,12,82160,68,53160,44
88,chebyshev_sqrt_decor_32,"(0.79, 2.36)",2.240307e-08,0.01108,24,4,66968,50,36624,58
140,taylor_sqrt_naive_32,"(0.79, 2.36)",8.709681e-03,0.01487,6,7,48336,40,31416,26
142,taylor_sqrt_decor_32,"(0.79, 2.36)",8.709681e-03,0.00961,24,3,52504,46,34208,56
41,fourier_sqrt_32,"(0.79, 2.36)",2.336460e-02,0.00337,1,1,96384,8,24096,2


In [57]:
by_interval(df[df['Method'].str.contains('mul_inv')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
84,chebyshev_mul_inv_decor_32,"(0.79, 2.36)",4.649327e-07,0.01231,24,4,66968,50,36624,58
91,chebyshev_mul_inv_clenshaw_32,"(0.79, 2.36)",4.650617e-07,0.02961,10,12,82144,68,53152,44
81,chebyshev_mul_inv_naive_32,"(0.79, 2.36)",4.651152e-07,0.02814,10,12,82160,68,53160,44
150,taylor_mul_inv_decor_32,"(0.79, 2.36)",8.392917e-04,0.01080,24,3,52504,46,34208,56
138,taylor_mul_inv_naive_32,"(0.79, 2.36)",8.392919e-04,0.01471,6,7,48336,40,31416,26
40,fourier_mul_inv_32,"(0.79, 2.36)",3.054742e-02,0.00253,1,1,96384,8,24096,2


In [58]:
by_interval(df[df['Method'].str.contains('log')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
87,chebyshev_log_clenshaw_32,"(0.79, 2.36)",5.791410e-08,0.02876,10,12,82144,68,53152,44
86,chebyshev_log_naive_32,"(0.79, 2.36)",5.793039e-08,0.02723,10,12,82160,68,53160,44
80,chebyshev_log_decor_32,"(0.79, 2.36)",6.215700e-08,0.01101,24,4,66968,50,36624,58
141,taylor_log_naive_32,"(0.79, 2.36)",1.783266e-04,0.01582,6,7,48336,40,31416,26
151,taylor_log_decor_32,"(0.79, 2.36)",1.783267e-04,0.01021,24,3,52504,46,34208,56
39,fourier_log_32,"(0.79, 2.36)",3.955955e-02,0.00142,1,1,96384,8,24096,2


In [59]:
by_interval(df[df['Method'].str.contains('tanh')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
14,decor_tanh_32,"(-1.0, 2.5)",3.258099e-09,0.41969,184,59,3505632,534,1986528,496
69,chebyshev_tanh_decor_32,"(-1.0, 2.5)",9.318108e-05,0.01277,24,4,66968,50,36624,58
68,chebyshev_tanh_naive_32,"(-1.0, 2.5)",9.320868e-05,0.02487,10,12,82160,68,53160,44
74,chebyshev_tanh_clenshaw_32,"(-1.0, 2.5)",9.320870e-05,0.02866,10,12,82144,68,53152,44
32,fourier_tanh_32,"(-1.0, 2.5)",6.300809e-02,0.00403,1,1,96384,8,24096,2
131,taylor_tanh_naive_32,"(-1.0, 2.5)",1.508766e-01,0.00783,3,4,26592,22,16920,14
132,taylor_tanh_decor_32,"(-1.0, 2.5)",1.508767e-01,0.01127,24,3,45280,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
10,decor_tanh_32,"(-3.14, 3.14)",3.066312e-07,0.41932,184,59,3505632,534,1986528,496
54,chebyshev_tanh_decor_32,"(-3.14, 3.14)",4.513010e-03,0.01385,24,4,66968,50,36624,58
53,chebyshev_tanh_naive_32,"(-3.14, 3.14)",4.513022e-03,0.02673,10,12,82160,68,53160,44
59,chebyshev_tanh_clenshaw_32,"(-3.14, 3.14)",4.513022e-03,0.02536,10,12,82144,68,53152,44
27,fourier_tanh_32,"(-3.14, 3.14)",7.177069e-02,0.00282,1,1,96384,8,24096,2
122,taylor_tanh_decor_32,"(-3.14, 3.14)",1.857928e+00,0.01117,24,3,45280,46,34208,56
121,taylor_tanh_naive_32,"(-3.14, 3.14)",1.857928e+00,0.00804,3,4,26592,22,16920,14


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
22,decor_tanh_32,"(-5.0, 5.0)",0.001626,0.43142,184,59,3505632,534,1986528,496
107,chebyshev_tanh_naive_32,"(-5.0, 5.0)",0.021485,0.02498,10,12,82160,68,53160,44
113,chebyshev_tanh_clenshaw_32,"(-5.0, 5.0)",0.021485,0.02808,10,12,82144,68,53152,44
108,chebyshev_tanh_decor_32,"(-5.0, 5.0)",0.021485,0.01198,24,4,66968,50,36624,58
45,fourier_tanh_32,"(-5.0, 5.0)",0.072037,0.00363,1,1,96384,8,24096,2
158,taylor_tanh_decor_32,"(-5.0, 5.0)",9.070970,0.01100,24,3,45280,46,34208,56
157,taylor_tanh_naive_32,"(-5.0, 5.0)",9.070970,0.00788,3,4,26592,22,16920,14


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
18,decor_tanh_32,"(0.79, 2.36)",2.664217e-10,0.41890,184,59,3505632,534,1986528,496
89,chebyshev_tanh_naive_32,"(0.79, 2.36)",5.572365e-09,0.02903,10,12,82160,68,53160,44
97,chebyshev_tanh_clenshaw_32,"(0.79, 2.36)",5.586335e-09,0.02913,10,12,82144,68,53152,44
90,chebyshev_tanh_decor_32,"(0.79, 2.36)",1.156482e-08,0.01215,24,4,66968,50,36624,58
147,taylor_tanh_naive_32,"(0.79, 2.36)",1.808822e-03,0.00901,3,4,26592,22,16920,14
143,taylor_tanh_decor_32,"(0.79, 2.36)",1.808822e-03,0.00878,24,3,45280,46,34208,56
37,fourier_tanh_32,"(0.79, 2.36)",1.175506e-02,0.00317,1,1,96384,8,24096,2


In [60]:
by_interval(df[df['Method'].str.contains('polynomial')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
15,decor_polynomial_32,"(-1.0, 2.5)",1.308124e-08,0.00998,24,3,45280,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
11,decor_polynomial_32,"(-3.14, 3.14)",1.525825e-07,0.01103,24,3,45280,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
23,decor_polynomial_32,"(-5.0, 5.0)",7.916242e-07,0.00936,24,3,45280,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
19,decor_polynomial_32,"(0.79, 2.36)",1.342968e-08,0.00927,24,3,45280,46,34208,56


In [61]:
""" E2E apps """

# Until Codon Jupyter is fixed: Read the data from files
dump_folder = "dump"
dump_files = [
    "log_reg",
    "dti"
    ]
nbit_fs = [32]

df_data = {
    'Method': [],
    'Accuracy': [],
    'Runtime': [],
    'Bytes sent': [],
    'Requests sent': [],
    'Partitions count': [],
    'Truncations count': []
    }

for dump_file in dump_files:
    for nbit_f in nbit_fs:
        try:
            with open(f"{dump_folder}/{dump_file}_{nbit_f}.p", "rb") as f:
                data = pickle.load(f)
                for k, v in data.items():
                    if not k.endswith('_accuracy'):
                        continue
                    
                    k = k.replace('_accuracy', '')
                    accuracy = data[f"{k}_accuracy"][0]
                    runtime = round(data[f"{k}_time"][0], 5)
                    bytes_sent = int(data[f"{k}_bytes_sent"][0])
                    send_requests = int(data[f"{k}_send_requests"][0])
                    partitions_count = int(data[f"{k}_partitions_count"][0])
                    truncations_count = int(data[f"{k}_truncations_count"][0])
                    
                    df_data['Method'].append(f"{k}_{nbit_f}")
                    
                    df_data['Accuracy'].append(accuracy)
                    df_data['Runtime'].append(runtime)
                    df_data['Bytes sent'].append(bytes_sent)
                    df_data['Requests sent'].append(send_requests)
                    df_data['Partitions count'].append(partitions_count)
                    df_data['Truncations count'].append(truncations_count)
        except FileNotFoundError:
            print(f"Could not find {dump_folder}/{dump_file}_{nbit_f}.p")

e2e_df = pd.DataFrame(df_data)

In [62]:
e2e_df[e2e_df['Method'].str.contains('binary')].sort_values(by='Accuracy', ascending=False)

,Method,Accuracy,Runtime,Bytes sent,Requests sent,Partitions count,Truncations count
4,log_reg_binary_decor_32,0.9375,0.21557,1237072,6602,1841,1440
1,log_reg_binary_chebyshev_32,0.8750,0.03795,205392,922,201,260
2,log_reg_binary_fourier_32,0.6875,0.19390,148112,282,81,80
5,log_reg_binary_taylor_32,0.5000,0.02244,117552,482,101,140


In [63]:
e2e_df[e2e_df['Method'].str.contains('multinomial')].sort_values(by='Accuracy', ascending=False)

,Method,Accuracy,Runtime,Bytes sent,Requests sent,Partitions count,Truncations count
3,log_reg_multinomial_fourier_32,0.9375,0.56319,2458512,5122,2361,80
6,log_reg_multinomial_chebyshev_32,0.9375,0.40785,2623312,5762,2481,260
7,log_reg_multinomial_taylor_32,0.9375,0.39607,2506672,5562,2441,200
0,log_reg_multinomial_decor_32,0.7500,0.40416,2537552,6162,2801,120


In [64]:
e2e_df[e2e_df['Method'].str.contains('dti')].sort_values(by='Accuracy', ascending=False)

,Method,Accuracy,Runtime,Bytes sent,Requests sent,Partitions count,Truncations count
15,dti_linear_chebyshev_32,0.708333,168.88176,2403710984,5646,2201,498
8,dti_tanh_chebyshev_32,0.583333,169.70991,2372641816,5876,1885,969
13,dti_sigmoid_fourier_32,0.562500,178.18849,2401271016,9124,3299,1180
9,dti_tanh_fourier_32,0.520833,173.81166,2371618536,4564,1639,600
10,dti_tanh_decor_32,0.520833,182.78921,2490082072,24736,9101,2978
11,dti_tanh_taylor_32,0.500000,173.72765,2371038256,4974,1680,723
12,dti_sigmoid_chebyshev_32,0.500000,176.92023,2402294296,10436,3545,1549
14,dti_sigmoid_taylor_32,0.500000,177.04375,2400690736,9534,3340,1303
16,dti_sigmoid_decor_32,0.479167,187.79492,2519734552,29296,10761,3558
